# BM25 & Kiwi(형태소) 테스트

`uv add kiwipiepy rank_bm25`

In [ ]:
documents = [
    "자연어 처리는 컴퓨터가 인간의 언어를 이해하는 기술입니다.",
    "Kiwi는 빠르고 정확한 한국어 형태소 분석기입니다.",
    "BM25는 문서 검색 및 정보 검색 시스템에서 자주 쓰이는 알고리즘입니다.",
    "파이썬을 사용하면 자연어 처리와 텍스트 마이닝을 쉽게 할 수 있습니다."
]

In [ ]:
from kiwipiepy import Kiwi
from rank_bm25 import BM25Okapi

def tokenize_str(doc: str):
    kiwi = Kiwi()
    result = []

    for token in kiwi.tokenize(doc):
        # 일반명사, 보통명사, 동사, 형용사, 외국어/외래어
        if token.tag in ('NNG', 'NNP', 'VV', 'VA', 'SL'):
            result.append(token.form)

    return result

kr_corpus = [tokenize_str(doc) for doc in documents]
print(kr_corpus)

In [ ]:
bm25 = BM25Okapi(kr_corpus)

query = '검색'

tk_query = tokenize_str(query)
print(tk_query)
doc_scores = bm25.get_scores(tk_query)

for i, score in enumerate(doc_scores):
    print(f"문서 {i} 점수: {score:.4f} | 원문: {documents[i]}")


## `md` -> `chunk` -> 형태소분리 -> `BM25` 검색기 -> 검색
핵심은 키워드(형태소) 매칭이다!

In [ ]:
import sys
from pathlib import Path

# 현재 노트북(src/notebooks) 위치 기준으로 상위 프로젝트 루트(08_RAG2)를 sys.path에 추가
PROJECT_ROOT = Path.cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [ ]:
# 이제 정상적으로 import 가능합니다.
from src.load_split import process_llamaparse_markdown

file_path = PROJECT_ROOT / "src/parsed_data/output.md"

with open(file_path, encoding="utf-8") as f:
    markdown_text = f.read()

chunks = process_llamaparse_markdown(markdown_text, source_name='nia-2026-ai.pdf')

In [ ]:
# 3. Kiwi 형태소 분석기 인스턴스 및 토큰화 함수
kiwi = Kiwi()

def tokenize_str(doc: str) -> list[str]:
    result = []
    for token in kiwi.tokenize(doc):
        # 일반명사(NNG), 고유명사(NNP), 동사(VV), 형용사(VA), 외국어(SL), 숫자(SN) 등
        if token.tag in ("NNG", "NNP", "VV", "VA", "SL", "SN"):
            result.append(token.form)
    return result

# 4. 전체 Chunk 문서(page_content) 형태소 토큰화 및 BM25 코퍼스 구축
tokenized_corpus = [tokenize_str(doc.page_content) for doc in chunks]
bm25 = BM25Okapi(tokenized_corpus)
print("BM25 인덱싱 완료!")

In [49]:
import numpy as np

# 5. 검색 함수 정의 및 테스트
def search_bm25(query: str, top_k: int = 3):
    # 쿼리 토큰화
    tk_query = tokenize_str(query)
    print(f"[검색어 형태소 분석 결과]: {tk_query}")
    
    # 각 청크별 BM25 점수 계산
    scores = bm25.get_scores(tk_query)
    
    # 점수가 높은 순으로 정렬 (내림차순)
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    print(f"\n=== '{query}' 검색 결과 (Top {top_k}) ===")
    for rank, idx in enumerate(top_indices, start=1):
        score = scores[idx]
        if score == 0:
            print(f"\n[{rank}] 점수: 0 (더 이상 일치하는 문서 없음)")
            break
            
        doc = chunks[idx]
        print(f"\n[{rank}] 점수: {score:.4f} | 메타데이터: {doc.metadata}")
        print("-" * 50)
        print(doc.page_content)


In [50]:
# 형태소 저장 -> bm25검색 테스트

search_bm25("노동 집약적", top_k=10)

[검색어 형태소 분석 결과]: ['노동', '집약']

=== '노동 집약적' 검색 결과 (Top 10) ===

[1] 점수: 5.4415 | 메타데이터: {'source': 'nia-2026-ai.pdf', 'type': 'text', 'h1': '트렌드 3 AI가 현실 세계로, 산업 현장에서 시작되는 피지컬 AI 혁신', 'h2': '핵심내용 및 전망', 'h3': '생산성 및 효율성 극대화'}
--------------------------------------------------
### 생산성 및 효율성 극대화  
* 제조, 물류 등 산업 현장에서 로봇과 AI가 24시간 365일 작업을 지속함으로써, 설비 가동률과 총 생산량을 극대화  
* 노동 집약적인 물류 및 제조 분야에서 인력 의존도를 줄여 구조적인 비용 절감을 실현하며, 장기적으로 기업의 경쟁력을 제고  
**정책 방향**
* (전 산업 피지컬 AI 확산) 조선 AX, 국방 AX 등 피지컬 AI 기반 산업별 전략 추진을 위해 민관 협력체계를 본격 가동하고, 전 산업으로 피지컬 AI 적용 범위를 확대
* (지역산업 AI 혁신 엔진 가동) 지역 AX 프로젝트와 연계하여 제조·물류 등 강점 분야에 대한 피지컬 AI 실증 테스트베드를 구축하고 조기에 산업 생산성 향상을 입증
* (피지컬 AI 인재·생태계 조성) 제조 현장 등 성장 잠재력이 높은 분야의 피지컬 AI 확산을 위해, 산업 현장에 특화된 피지컬 AI 전문 인력을 전략적으로 확보하고 육성  
32 한국지능정보사회진흥원 NIA가 전망한 2026년 12대 AIㆍ디지털 트렌드 33  
---  
OBEA:AIEO  
트렌드 4

[2] 점수: 5.0587 | 메타데이터: {'source': 'nia-2026-ai.pdf', 'type': 'text', 'h1': 'NIA가 전망한 2026년 12대 AI·디지털 트렌드', 'h2': 'AI발(發) 노동시장의 대격변과 양극화 심화 IV'}
----------------------------------